# Neural Network Modeling: Feed-forward Neural Network Classifier (Loan Data)

In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## Load and preprocess the loan dataset

In [11]:
loan_df = pd.read_excel("loan.xlsx")
loan_df.head()

,Sex,Age,Time_at_address,Res_status,Telephone,Occupation,Job_status,Time_employed,Time_bank,Liab_ref,Acc_ref,Home_Expn,Balance,Decision
0,M,50.750000,0.585,owner,given,unemploye,unemploye,0,0,f,given,145,0,reject
1,M,19.670000,10.000,rent,not_given,labourer,governmen,0,0,t,given,140,0,reject
2,F,52.830002,15.000,owner,given,creative_,private_s,5,14,f,given,0,2200,accept
3,M,22.670000,2.540,rent,not_given,creative_,governmen,2,0,f,given,0,0,accept
4,M,29.250000,13.000,owner,given,driver,governmen,0,0,f,given,228,0,reject


In [12]:
# Target encoding
target_col = "Decision"
le = LabelEncoder()
y = le.fit_transform(loan_df[target_col])

# Features
X_raw = loan_df.drop(columns=[target_col])

# One-hot encode categorical features
X = pd.get_dummies(X_raw, drop_first=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print("X shape:", X.shape)
print("Train:", X_train_sc.shape, "Test:", X_test_sc.shape)
print("Classes:", list(le.classes_))

X shape: (429, 28)
Train: (321, 28) Test: (108, 28)
Classes: ['accept', 'reject']


## Baseline feed-forward neural network (MLP)

In [13]:
def build_model(input_dim: int, layers=(16, 8), activation="relu", dropout=0.0):
    model = Sequential()
    # first hidden layer with input_dim
    model.add(Dense(layers[0], activation=activation, input_shape=(input_dim,)))
    if dropout > 0:
        model.add(Dropout(dropout))
    # additional hidden layers
    for units in layers[1:]:
        model.add(Dense(units, activation=activation))
        if dropout > 0:
            model.add(Dropout(dropout))
    # output layer for binary classification
    model.add(Dense(1, activation="sigmoid"))
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model

baseline = build_model(input_dim=X_train_sc.shape[1], layers=(16, 8), activation="relu", dropout=0.0)
baseline.summary()

/Users/aakrutighatole/.pyenv/versions/3.9.17/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_53 (Dense)                │ (None, 16)             │           464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_54 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_55 (Dense)                │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 609 (2.38 KB)

 Trainable params: 609 (2.38 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
early_stop = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

history = baseline.fit(
    X_train_sc, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=0
)

y_pred = (baseline.predict(X_test_sc) >= 0.5).astype(int).ravel()
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
Baseline Accuracy: 0.7685185185185185
              precision    recall  f1-score   support

      accept       0.71      0.81      0.76        48
      reject       0.83      0.73      0.78        60

    accuracy                           0.77       108
   macro avg       0.77      0.77      0.77       108
weighted avg       0.78      0.77      0.77       108

Confusion Matrix:
 [[39  9]
 [16 44]]


## Architecture search (best-performing model)



In [15]:
from itertools import product

configs = [
    # (layers, activation, dropout)
    ((8,), "relu", 0.0),
    ((16,), "relu", 0.0),
    ((32,), "relu", 0.0),
    ((16, 8), "relu", 0.0),
    ((32, 16), "relu", 0.0),
    ((64, 32), "relu", 0.2),
    ((32, 16, 8), "relu", 0.2),
    ((16, 8), "tanh", 0.0),
    ((32, 16), "tanh", 0.0),
]

best = {"val_acc": -1, "config": None, "model": None}

for layers, act, dr in configs:
    model = build_model(input_dim=X_train_sc.shape[1], layers=layers, activation=act, dropout=dr)
    es = EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True)
    hist = model.fit(
        X_train_sc, y_train,
        epochs=120,
        batch_size=16,
        validation_split=0.2,
        callbacks=[es],
        verbose=0
    )
    val_acc = max(hist.history.get("val_accuracy", [0]))
    if val_acc > best["val_acc"]:
        best.update({"val_acc": val_acc, "config": (layers, act, dr), "model": model})

print("Best config:", best["config"])
print("Best val_accuracy:", best["val_acc"])

/Users/aakrutighatole/.pyenv/versions/3.9.17/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best config: ((16,), 'relu', 0.0)
Best val_accuracy: 0.8153846263885498


In [16]:
best_model = best["model"]
y_pred_best = (best_model.predict(X_test_sc) >= 0.5).astype(int).ravel()

print("Test Accuracy (Best):", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best, target_names=le.classes_))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Test Accuracy (Best): 0.7037037037037037
              precision    recall  f1-score   support

      accept       0.66      0.69      0.67        48
      reject       0.74      0.72      0.73        60

    accuracy                           0.70       108
   macro avg       0.70      0.70      0.70       108
weighted avg       0.71      0.70      0.70       108

Confusion Matrix:
 [[33 15]
 [17 43]]
